# 🩺 Şeker Hastalığı (Diyabet) Teşhisi — Makine Öğrenmesi Projesi

**Proje Özeti:**
- **Problem:** Kişinin sağlık ölçümlerine göre diyabetli olup olmadığını tahmin etmek
- **Problem Türü:** İkili Sınıflandırma (Diyabetli: 1 / Sağlıklı: 0)
- **Veri Seti Kaynağı:** Kaggle — Pima Indians Diabetes Database
- **Kullanılan Algoritmalar:** Lojistik Regresyon, Karar Ağacı, Rastgele Orman
- **Proje Amacı:** Gerçek sağlık verileri üzerinde uçtan uca makine öğrenmesi sınıflandırma modeli kurmak

## 📦 Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

sns.set_theme(style='whitegrid', palette='muted')
print('Kütüphaneler başarıyla yüklendi ✅')

## 📁 Bölüm 1 — Veri Seti Tanıtımı

Veri seti 768 kişinin sağlık ölçümlerini içermektedir. Her satır bir kişiyi, her sütun bir özelliği temsil eder.

| Sütun | Açıklama |
|---|---|
| Pregnancies | Gebelik sayısı |
| Glucose | Kan şekeri değeri |
| BloodPressure | Kan basıncı |
| SkinThickness | Deri kalınlığı |
| Insulin | İnsülin seviyesi |
| BMI | Vücut kitle indeksi |
| DiabetesPedigreeFunction | Aile diyabet geçmişi skoru |
| Age | Yaş |
| **Outcome** | **Hedef: 0 = Sağlıklı, 1 = Diyabetli** |

In [ ]:
df = pd.read_csv('diabetes.csv')

print(f'Satır sayısı  : {df.shape[0]}')
print(f'Sütun sayısı  : {df.shape[1]}')
print()
print('İlk 5 satır:')
df.head()

In [ ]:
print('Hedef sınıf dağılımı (0=Sağlıklı, 1=Diyabetli):')
print(df['Outcome'].value_counts())
print()
print('Sütun veri tipleri:')
print(df.dtypes)

## 🔍 Bölüm 2 — Keşifsel Veri Analizi (EDA)

### 2.1 Eksik Veri Analizi
Bazı sütunlarda sıfır (0) değeri gerçekte **eksik veriyi** temsil eder. Örneğin kan şekeri veya kan basıncının 0 olması tıbben mümkün değildir.

In [ ]:
sifir_olamaz = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Mantıksal olarak 0 olamayacak sütunlardaki sıfır sayıları:')
print((df[sifir_olamaz] == 0).sum())

# 0 değerlerini NaN ile değiştir, medyan ile doldur
df[sifir_olamaz] = df[sifir_olamaz].replace(0, np.nan)
df[sifir_olamaz] = df[sifir_olamaz].fillna(df[sifir_olamaz].median())

print()
print('Eksik değerler medyan ile dolduruldu ✅')

### 2.2 Temel İstatistikler

In [ ]:
df.describe().round(2)

### 2.3 Aykırı Değer Analizi (IQR Yöntemi)

IQR (Çeyrekler Arası Aralık) yöntemiyle her sütundaki aykırı değer sayısı hesaplanmıştır.

In [ ]:
print('Aykırı değer sayıları (IQR yöntemi):')
for col in df.columns[:-1]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    aykiri = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col:30s}: {aykiri} aykırı değer')

### 2.4 Görselleştirme — Hedef Sınıf Dağılımı

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
df['Outcome'].value_counts().plot.pie(
    labels=['Sağlıklı (0)', 'Diyabetli (1)'],
    autopct='%1.1f%%',
    colors=['#4CAF50', '#F44336'],
    startangle=90, ax=ax
)
ax.set_ylabel('')
ax.set_title('Hedef Sınıf Dağılımı')
plt.tight_layout()
plt.show()

### 2.5 Görselleştirme — Özelliklerin Dağılımı

In [ ]:
df.hist(bins=20, figsize=(12, 8), color='#5C85D6', edgecolor='white')
plt.suptitle('Özelliklerin Dağılımı', fontsize=14)
plt.tight_layout()
plt.show()

### 2.6 Görselleştirme — Korelasyon Isı Haritası

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Korelasyon Isı Haritası')
plt.tight_layout()
plt.show()

### 2.7 Görselleştirme — Kan Şekeri ve Vücut Kitle İndeksi İlişkisi

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Glucose', y='BMI',
                hue='Outcome', palette={0: '#4CAF50', 1: '#F44336'}, alpha=0.7)
plt.title('Kan Şekeri - Vücut Kitle İndeksi İlişkisi')
plt.legend(title='Durum', labels=['Sağlıklı', 'Diyabetli'])
plt.tight_layout()
plt.show()

## 🤖 Bölüm 3 — Model Kurma

Veri seti **%80 eğitim / %20 test** olarak ikiye ayrılmıştır. Üç farklı algoritma denenmiştir:

1. **Lojistik Regresyon** — Basit ve hızlı, başlangıç referans modeli
2. **Karar Ağacı** — Dal dal karar vererek sınıflandırır
3. **Rastgele Orman** — 100 karar ağacının ortak kararı, en güçlü model

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

olcekleyici = StandardScaler()
X_egitim_o = olcekleyici.fit_transform(X_egitim)
X_test_o   = olcekleyici.transform(X_test)

print(f'Eğitim seti boyutu : {X_egitim.shape[0]} satır')
print(f'Test seti boyutu   : {X_test.shape[0]} satır')

In [ ]:
modeller = {
    'Lojistik Regresyon': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Karar Ağacı'       : (DecisionTreeClassifier(max_depth=5, random_state=42), False),
    'Rastgele Orman'    : (RandomForestClassifier(n_estimators=100, random_state=42), False),
}

sonuclar = {}

for isim, (model, olcekli) in modeller.items():
    model.fit(X_egitim_o if olcekli else X_egitim,
              y_egitim)
    y_tahmin = model.predict(X_test_o if olcekli else X_test)
    sonuclar[isim] = {
        'Doğruluk'   : accuracy_score(y_test, y_tahmin),
        'Hassasiyet' : precision_score(y_test, y_tahmin),
        'Duyarlılık' : recall_score(y_test, y_tahmin),
        'F1 Skoru'   : f1_score(y_test, y_tahmin),
        'y_tahmin'   : y_tahmin,
    }
    print(f'{isim} modeli eğitildi ✅')

## 📊 Bölüm 4 — Model Değerlendirme

### 4.1 Model Karşılaştırma Tablosu

In [ ]:
tablo = pd.DataFrame(
    {k: {m: round(v, 4) for m, v in sonuclar[k].items() if m != 'y_tahmin'}
     for k in sonuclar}
).T

print('MODEL KARŞILAŞTIRMASI')
print('=' * 60)
tablo

In [ ]:
en_iyi = max(sonuclar, key=lambda k: sonuclar[k]['Doğruluk'])
print(f'✅ En iyi model: {en_iyi}')
print(f'   Doğruluk oranı: %{sonuclar[en_iyi]["Doğruluk"]*100:.2f}')

### 4.2 Karışıklık Matrisi (En İyi Model)

In [ ]:
cm = confusion_matrix(y_test, sonuclar[en_iyi]['y_tahmin'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sağlıklı', 'Diyabetli'],
            yticklabels=['Sağlıklı', 'Diyabetli'])
plt.title(f'Karışıklık Matrisi — {en_iyi}')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek Değer')
plt.tight_layout()
plt.show()

### 4.3 Model Performans Karşılaştırma Grafiği

In [ ]:
tablo[['Doğruluk', 'Hassasiyet', 'Duyarlılık', 'F1 Skoru']].plot(
    kind='bar', figsize=(9, 5), edgecolor='white'
)
plt.title('Model Performans Karşılaştırması')
plt.ylabel('Skor')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 4.4 Rastgele Orman — Özellik Önem Sıralaması

Hangi özelliğin diyabet tahminine en çok katkı sağladığını gösterir.

In [ ]:
rf_model = modeller['Rastgele Orman'][0]
onem = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 5))
onem.plot(kind='barh', color='#5C85D6')
plt.title('Rastgele Orman — Özellik Önem Sıralaması')
plt.xlabel('Önem Skoru')
plt.tight_layout()
plt.show()

## ✅ Sonuç

Bu projede şeker hastalığı teşhisi için üç farklı makine öğrenmesi algoritması uygulanmış ve karşılaştırılmıştır.

- **En başarılı model:** Rastgele Orman (%70.13 doğruluk)
- **En belirleyici özellik:** Kan şekeri (Glucose) değeri
- **Önemli bulgu:** Veri setinde sınıf dengesizliği mevcuttur (%71 sağlıklı, %29 diyabetli). Bu nedenle F1 skoru, doğruluk oranına ek olarak değerlendirmeye alınmıştır.